In [1]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [2]:
import numpy as np
import torch

from pathlib import Path

from storyforge.config import load_config
from storyforge.training import load_initial_model
from storyforge.optimization import build_optimizer
from storyforge.evaluation import evaluate
from storyforge.metric import record_metrics

In [3]:
config = load_config("configs/default.toml")

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [5]:
model = load_initial_model(config, device)
optimizer = build_optimizer(model, config)

In [6]:
train_tokens = np.load(
    config["encoding"]["train_path"],
    mmap_mode="r",
)

In [7]:
val_tokens = np.load(
    config["encoding"]["validation_path"],
    mmap_mode="r",
)

In [8]:
rng = np.random.default_rng(config["seed"])

In [9]:
print("Device:", device)
print("Loaded the initial model.")

Device: mps
Loaded the initial model.


In [10]:
metrics_path = Path(config["logging"]["metrics_path"])

In [11]:
if metrics_path.exists():
    raise FileExistsError(
        "A metrics file already exists. "
        "Use a fresh metrics path for a new training run."
    )

In [12]:
initial_validation_loss = evaluate(
    model,
    val_tokens,
    config,
    device,
)

record_metrics(
    metrics_path,
    {
        "step": 0,
        "processed_tokens": 0,
        "train_loss": None,
        "val_loss": initial_validation_loss,
        "learning_rate": None,
        "elapsed_seconds": 0.0,
    },
)

In [13]:
print("Initial validation loss:", initial_validation_loss)
print("Saved initial measurement:", metrics_path)

Initial validation loss: 9.087669491767883
Saved initial measurement: runs/baseline/metrics.jsonl


In [14]:
from storyforge.checkpoint import save_checkpoint

In [15]:
config = load_config("configs/default.toml")

In [16]:
best_val_loss = initial_validation_loss

In [17]:
save_checkpoint(
    path=config["checkpoints"]["best_path"],
    model=model,
    optimizer=optimizer,
    config=config,
    step=0,
    processed_tokens=0,
    val_loss=best_val_loss,
)

In [18]:
print("Current best validation loss:", best_val_loss)
print("Saved:", config["checkpoints"]["best_path"])

Current best validation loss: 9.087669491767883
Saved: runs/baseline/best.pt


#### **Final Training**

In [20]:
from storyforge.runner import run_training
from storyforge.plotting import plot_loss_curves

In [21]:
config = load_config("configs/default.toml")

In [22]:
model = load_initial_model(config, device)
optimizer = build_optimizer(model, config)
rng = np.random.default_rng(config["seed"])

In [23]:
summary = run_training(
    model=model,
    optimizer=optimizer,
    train_tokens=train_tokens,
    val_tokens=val_tokens,
    config=config,
    rng=rng,
    device=device,
    initial_validation_loss=initial_validation_loss,
)


Update 1/1526 | tokens 16,384 | train 9.0729 | validation None
Update 10/1526 | tokens 163,840 | train 8.3770 | validation None
Update 20/1526 | tokens 327,680 | train 7.9260 | validation None
Update 30/1526 | tokens 491,520 | train 7.4384 | validation None
Update 40/1526 | tokens 655,360 | train 6.8535 | validation None
Update 50/1526 | tokens 819,200 | train 6.3120 | validation None
Update 60/1526 | tokens 983,040 | train 5.7404 | validation None
Update 70/1526 | tokens 1,146,880 | train 5.3398 | validation None
Update 80/1526 | tokens 1,310,720 | train 5.0168 | validation None
Update 90/1526 | tokens 1,474,560 | train 4.7266 | validation None
Update 100/1526 | tokens 1,638,400 | train 4.5542 | validation 4.261678904294968
Update 110/1526 | tokens 1,802,240 | train 4.4265 | validation None
Update 120/1526 | tokens 1,966,080 | train 4.2828 | validation None
Update 130/1526 | tokens 2,129,920 | train 4.1988 | validation None
Update 140/1526 | tokens 2,293,760 | train 4.1976 | validatio

In [24]:
plot_path = plot_loss_curves(config)

In [25]:
print("Best validation loss:", summary["best_val_loss"])
print("Best validation perplexity:", summary["best_val_perplexity"])
print("Training tokens/second:", summary["training_tokens_per_second"])
print("Saved loss plot:", plot_path)

Best validation loss: 2.2215647846460342
Best validation perplexity: 9.221749634043567
Training tokens/second: 32235.517717989984
Saved loss plot: runs/baseline/loss_curves.png
